# Chapter 5 — Codex: Configuration, Sandboxing and CI · Homework (self-test)

**Goal:** check that you can configure Codex for a team and run it safely in CI, without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 75–90 minutes
- Estimated API cost: **$0**. Everything here is offline; no exercise calls the Codex CLI, the SDK or the API.
- Prerequisites: repo venv from [00 — Prerequisites](../../00-prerequisites/README.md)

### How to use this notebook

1. Answer the quiz in Part A by filling in the `answers` dict, then run the grader.
2. Implement each exercise in Part B (replace the `raise NotImplementedError`), then run its check cell.
3. Write your answer to the Part C scenario before opening the rubric.
4. Run the scorecard. Solutions are at the very bottom; try everything first.

**Scoring:** quiz score (12 questions) and exercises passed (5). The pass mark is **72%** on the combined score: the same bar as the Claude path, which mirrors the Claude exam's 720/1000.

In [ ]:
import hashlib
import json
import os
import tomllib

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

RESULTS: dict[str, bool] = {f"ex{i}": False for i in range(1, 6)}
QUIZ_SCORE: dict[str, int] = {"correct": 0, "total": 12}

def run_check(name: str, fn) -> None:
    """Run a checker without ever crashing the notebook."""
    try:
        fn()
        RESULTS[name] = True
        print(f"✓ Exercise {name[2:]} passed")
    except NotImplementedError:
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]}: not implemented yet (replace the NotImplementedError)")
    except AssertionError as e:
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]}: {e}")
    except Exception as e:  # noqa: BLE001 - any bug in learner code is reported, not raised
        RESULTS[name] = False
        print(f"✗ Exercise {name[2:]} raised {type(e).__name__}: {e}")
print("ready")

## Part A — Quiz (12 scenario questions)

Pick one letter per question. Each question has exactly one best answer under the **current** Codex docs.

**Q1.** You run Codex in `repo/apps/web/`. `~/.codex/` contains a non-empty `AGENTS.override.md` **and** an `AGENTS.md`. The repo has `repo/AGENTS.md` and `repo/apps/web/AGENTS.md`. Which files form the instruction chain, in order?

- A) `~/.codex/AGENTS.md`, `~/.codex/AGENTS.override.md`, `repo/AGENTS.md`, `repo/apps/web/AGENTS.md`
- B) `~/.codex/AGENTS.override.md`, `repo/AGENTS.md`, `repo/apps/web/AGENTS.md`
- C) `~/.codex/AGENTS.md`, `repo/AGENTS.md`, `repo/apps/web/AGENTS.md` (overrides only apply inside projects)
- D) `repo/apps/web/AGENTS.md` only, because the closest file replaces all others

**Q2.** After a Codex upgrade, one developer's Codex refuses to start. Their `~/.codex/config.toml` contains `approval_policy = "untrusted"` because they want every command approved in a legacy repo. What is the best fix?

- A) Change it to `approval_policy = "on-failure"`
- B) Reinstall the CLI to clear the cached config
- C) Remove the retired value; use `sandbox_mode = "read-only"` + `approval_policy = "on-request"`, or keep strict command approvals by marking that project `trust_level = "untrusted"` under `[projects."<path>"]`
- D) Set `approval_policy = "never"` and `sandbox_mode = "danger-full-access"` so nothing prompts

**Q3.** A pre-merge job must produce review findings as structured data and must not modify the checkout. Which invocation fits best?

- A) `codex exec --json --sandbox read-only --output-schema review.schema.json "Review the diff"`, then parse the final `agent_message`
- B) `codex exec --full-auto "Review the diff and respond only in JSON"`, then parse stdout with a regex
- C) `codex "Review the diff" < /dev/null`
- D) `codex exec --sandbox danger-full-access --json "Review the diff"`

**Q4.** CI should use `gpt-6-luna` with a read-only sandbox and no approval prompts. Developers' laptops should keep `gpt-6.1-sol` and `workspace-write`. What is the cleanest setup on the runner and laptops?

- A) Add `[profiles.ci]` to `~/.codex/config.toml` and set `profile = "ci"` on the runner
- B) Put the CI settings in the repo's `.codex/config.toml`
- C) Write the CI settings into `AGENTS.md` under a "## CI" heading
- D) Create `~/.codex/ci.config.toml` with top-level keys and run `codex exec --profile ci ...` in CI

**Q5.** You clone a third-party repo that ships `.codex/config.toml` (with `openai_base_url = "https://proxy.example.dev/v1"`) and `.codex/hooks.json`. You start Codex and do **not** mark the project as trusted. What happens?

- A) Both load, because any Git repo is trusted by default
- B) Codex skips the project `.codex/` layers (config, hooks and rules). Even in a trusted project, `openai_base_url` is ignored at the project layer.
- C) The config loads, but hooks wait for `/hooks` review
- D) `openai_base_url` applies immediately; only hooks are blocked

**Q6.** A teammate created `.codex/skills/migrate/SKILL.md` with valid `name` and `description`, but Codex never lists the skill. What fixes it?

- A) Move it to `.agents/skills/migrate/SKILL.md`
- B) Add `@.codex/skills/migrate/SKILL.md` to the root `AGENTS.md`
- C) Convert it to `~/.codex/prompts/migrate.md` and call `/prompts:migrate`
- D) Rename `SKILL.md` to `skill.toml`

**Q7.** In `workspace-write`, a task fails because `npm install` cannot reach the registry. Security wants autonomy to stay bounded. What do you do?

- A) Run the team with `--dangerously-bypass-approvals-and-sandbox`
- B) Set `approval_policy = "never"` so Codex stops asking
- C) Grant scoped network access: approve the escalation, or set `[sandbox_workspace_write] network_access = true` with the `network_proxy` feature and an allow rule for the registry domain
- D) Add "you may use the network for npm" to `AGENTS.md`

**Q8.** You want Codex to auto-fix failing CI on GitHub and open a PR. Which design follows the documented security guidance?

- A) One job with `contents: write`, `OPENAI_API_KEY` as a job-level env var, `codex exec --sandbox workspace-write`, then `git push`
- B) Use `safety-strategy: unsafe` so Codex can install system packages with sudo
- C) Ask `@codex review` to push fixes to the branch
- D) A Codex job with `contents: read` runs `openai/codex-action@v1` (`sandbox: workspace-write`) and uploads a patch artifact; a separate job with `contents: write` applies the patch and opens the PR without receiving the key

**Q9.** You add an MCP server for the Codex CLI and want the IDE extension and the ChatGPT desktop app to see it too, without repeating the setup. Where does it go?

- A) A `.mcp.json` file at the repo root
- B) An `[mcp_servers.<name>]` table in `~/.codex/config.toml` (or a trusted project's `.codex/config.toml`), for example via `codex mcp add`
- C) Run `codex mcp-server` and point the other clients at it
- D) A "## MCP servers" section in `AGENTS.md`

**Q10.** An internal Python service must start Codex threads on a build runner, run a review turn with a JSON Schema, and resume the thread later. Which option is correct?

- A) `pip install openai-codex-sdk` and import `codex_sdk`
- B) Start `codex mcp-server` and call it as an MCP tool from the Agents SDK
- C) `pip install openai-codex`; `from openai_codex import Codex, Sandbox`; `codex.thread_start(...)`, `thread.run(..., output_schema=...)`, `codex.thread_resume(thread_id)`
- D) `from agents import Agent, Runner` with a shell tool, because the Agents SDK *is* Codex

**Q11.** A developer asks Codex in the CLI: "Migrate the billing module to the new payments client." Codex edits 14 files, says it's done, and stops. CI then fails on 3 tests, and the reviewer can't tell what Codex ran. Which change to the *task setup* would most likely have prevented this?

- A) Start with `/plan`, then set `/goal Migrate billing to the new payments client; keep the public API unchanged; done when pytest tests/billing and make lint pass`, and ask for the commands run with exit codes plus anything not verified
- B) Raise `model_reasoning_effort` to the maximum and repeat the same prompt
- C) Add "Always run all tests" to `~/.codex/AGENTS.md` on the developer's laptop
- D) Re-run the task with `--dangerously-bypass-approvals-and-sandbox` so Codex can run any command it needs

**Q12.** Three issues remain after a Codex change: (1) `parse_date` ignores time zones, (2) `parse_date` rejects ISO week dates, and (3) the monthly report, which calls `parse_date`, shows the wrong month at boundaries. There is also an unrelated typo in the CLI help. How do you send the follow-up work?

- A) Four separate messages, one per issue, so each diff stays small
- B) One message with all four issues, so Codex sees everything at once
- C) Ask Codex to "fix the date bugs properly", then mention the typo afterwards
- D) One detailed message for issues 1–3 (they interact through `parse_date`) with a failing test case for each, then the typo as a separate message

In [ ]:
# Fill in one letter per question: "A", "B", "C" or "D".
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
    "q11": None, "q12": None,
}

In [ ]:
ANSWER_KEY = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'c85dcf1566dc5c0487a6fe1611c41ba32751e8080b552a419521a39c7fee21b2', 'q3': '2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3', 'q4': 'b634a8333266441f30307b2d3392d77b7b3bcf69bc053cabed2c65d2671fc12e', 'q5': 'fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c', 'q6': 'ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522', 'q7': '242d38a9a6e8adda29d627e4ede1ef3c7395ee0112db33f8f0b06b45d9a66887', 'q8': '85b91a1895880cedf03bb98aa9ebfb3c730c838ec5d8f59ec2d1d86bcb936ced', 'q9': 'b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e', 'q10': 'd6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5', 'q11': 'acb9b8f6dd2a3e57b9476b90caf32ab8ad47a11c4aea20d95df7aaad0d81238c', 'q12': '33d12c43f8f028fb844551369083e2b9daa804b371ce6b8835f63a07a5cac42e'}  # sha256 of "qid:LETTER", so the answers are not visible

def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid, digest in ANSWER_KEY.items():
        letter = (answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        correct += ok
        print(f"{qid:>4}: {letter or '-':2} {'✓' if ok else '✗'}")
    QUIZ_SCORE["correct"] = correct
    print(f"Quiz score: {correct}/{len(ANSWER_KEY)} ({correct / len(ANSWER_KEY):.0%})")
    return correct

grade_quiz(answers)

## Part B — Coding exercises

All five are offline. Each check cell prints ✓ or a hint and never stops the notebook.

### Exercise 1 — Write a repository `AGENTS.md`

Return the text of a root `AGENTS.md` for a Python repo. Requirements:

- at least **3** bullet lines (`- ...`)
- mentions the test command `make test`
- a `## Code Review Rules` section with at least one rule that states a **safe path** (the words "Safe path")
- **no** `@path` import lines (Codex has no import syntax)
- at most **2,000 bytes** (it must load alongside other files under the size cap)

[Theory](README.md#52-agentsmd-discovery-hierarchy-and-override)

In [ ]:
def write_agents_md() -> str:
    # TODO: return the AGENTS.md text
    raise NotImplementedError

In [ ]:
def _check_ex1():
    text = write_agents_md()
    assert isinstance(text, str) and text.strip(), "return a non-empty string"
    assert len(text.encode("utf-8")) <= 2000, f"too long: {len(text.encode())} bytes (max 2,000)"
    lines = text.splitlines()
    assert sum(l.lstrip().startswith("- ") for l in lines) >= 3, "add at least 3 bullet lines"
    assert "make test" in text, "mention the test command `make test`"
    assert any(l.strip() == "## Code Review Rules" for l in lines), "add a `## Code Review Rules` heading"
    assert "safe path" in text.split("## Code Review Rules", 1)[1].lower(), "give a review rule a 'Safe path'"
    assert not any(l.strip().startswith("@") or " @./" in l or " @~/" in l for l in lines), \
        "remove @path imports: AGENTS.md has no import syntax"

run_check("ex1", _check_ex1)

### Exercise 2 — Resolve the `AGENTS.md` chain

Implement the documented discovery rules on an in-memory repo:

- `tree` maps a directory path relative to the project root (`""` is the root, `"services/payments"`, …) to `{filename: content}`. Missing directories have no files.
- `cwd` is relative to the root (`""` for the root itself).
- `global_files` is the content of `~/.codex` (`{filename: content}`).

Rules: global scope first (`AGENTS.override.md`, else `AGENTS.md`; first **non-empty** file only). The global file is loaded separately and does **not** use the byte budget. Then every directory from the root down to `cwd`: the first non-empty of `AGENTS.override.md`, `AGENTS.md`, then each name in `fallbacks`; at most one per directory. The project files share **one budget of `max_bytes`** (UTF-8 bytes). Each loaded file uses up its size. The file that crosses the budget is truncated but still counts as loaded. Once the budget is used up, no further files load. (This is the behavior of the current Codex source; see the README note in 5.2.)

Return labels: `"~/.codex/<name>"` for the global file, `"<dir>/<name>"` for project files (just `"<name>"` at the root).

[Theory](README.md#52-agentsmd-discovery-hierarchy-and-override)

In [ ]:
def resolve_chain(tree: dict[str, dict[str, str]], cwd: str, global_files: dict[str, str],
                  fallbacks: tuple[str, ...] = (), max_bytes: int = 32 * 1024) -> list[str]:
    # TODO
    raise NotImplementedError

In [ ]:
def _check_ex2():
    G = {"AGENTS.md": "global rules\n"}
    T = {
        "": {"AGENTS.md": "root rules\n"},
        "services/payments": {"AGENTS.md": "plain\n", "AGENTS.override.md": "payments override\n"},
        "services/search": {"AGENTS.md": "search\n"},
        "docs": {"AGENTS.md": "   \n"},
        "support": {"TEAM_GUIDE.md": "team guide\n"},
    }
    got = resolve_chain(T, "services/payments", G)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md", "services/payments/AGENTS.override.md"], \
        f"payments chain wrong: {got} (override wins in its directory; one file per directory)"
    got = resolve_chain(T, "", G)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md"], f"root chain wrong: {got}"
    got = resolve_chain(T, "services/search", {"AGENTS.override.md": "tmp\n", "AGENTS.md": "global\n"})
    assert got == ["~/.codex/AGENTS.override.md", "AGENTS.md", "services/search/AGENTS.md"], \
        f"global override not handled: {got}"
    got = resolve_chain(T, "docs", G)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md"], f"empty/whitespace files must be skipped: {got}"
    assert resolve_chain(T, "support", G) == ["~/.codex/AGENTS.md", "AGENTS.md"], "fallback names only count when listed"
    got = resolve_chain(T, "support", G, fallbacks=("TEAM_GUIDE.md",))
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md", "support/TEAM_GUIDE.md"], f"fallback not used: {got}"
    got = resolve_chain(T, "services/payments", {}, max_bytes=12)
    assert got == ["AGENTS.md", "services/payments/AGENTS.override.md"], \
        f"budget: 11 bytes used, 1 left -> the override loads (truncated): {got}"
    got = resolve_chain(T, "services/payments", {}, max_bytes=11)
    assert got == ["AGENTS.md"], f"budget: 11 bytes used, 0 left -> stop: {got}"
    big = {"AGENTS.md": "x" * 100}
    T2 = {"": {"AGENTS.md": "y" * 100}, "a": {"AGENTS.md": "z" * 100}}
    got = resolve_chain(T2, "a", big, max_bytes=150)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md", "a/AGENTS.md"], \
        f"the global file uses no budget, and the file that crosses the cap is truncated, not dropped: {got}"
    got = resolve_chain(T2, "a", big, max_bytes=100)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md"], f"budget used up exactly by the root file -> stop: {got}"
    got = resolve_chain(T2, "a", big, max_bytes=50)
    assert got == ["~/.codex/AGENTS.md", "AGENTS.md"], f"root truncated to 50 bytes, nothing left for a/: {got}"
    assert resolve_chain(T, "services/payments/api", G)[-1] == "services/payments/AGENTS.override.md", \
        "directories without files contribute nothing, but parents still count"

run_check("ex2", _check_ex2)

### Exercise 3 — Write `config.toml` and a CI profile file

1. `write_config_toml()` returns a **user** `~/.codex/config.toml` (TOML text) with:
   - a non-empty `model`
   - `approval_policy = "on-request"` and `sandbox_mode = "workspace-write"`
   - network **off** in the workspace-write sandbox (`[sandbox_workspace_write]` → `network_access = false`)
   - an MCP server named `docs` at `https://developers.openai.com/mcp` that CI must not run without (`required = true`)
   - no legacy profile settings (`profile` key or `[profiles.*]` tables)
2. `write_ci_profile()` returns `(filename, toml_text)` for the profile used as `codex exec --profile ci`: read-only sandbox, no approval prompts, model `gpt-6-luna`.

The checker parses both with `tomllib`. [Theory](README.md#53-configtoml-layers-precedence-profiles-and-trust)

In [ ]:
def write_config_toml() -> str:
    # TODO
    raise NotImplementedError

def write_ci_profile() -> tuple[str, str]:
    # TODO: return (filename, toml_text)
    raise NotImplementedError

In [ ]:
def _check_ex3():
    cfg = tomllib.loads(write_config_toml())
    assert isinstance(cfg.get("model"), str) and cfg["model"], "set a non-empty `model`"
    assert cfg.get("approval_policy") == "on-request", "approval_policy should be 'on-request'"
    assert cfg.get("sandbox_mode") == "workspace-write", "sandbox_mode should be 'workspace-write'"
    sww = cfg.get("sandbox_workspace_write", {})
    assert sww.get("network_access") is False, "set [sandbox_workspace_write] network_access = false"
    docs = cfg.get("mcp_servers", {}).get("docs", {})
    assert docs.get("url") == "https://developers.openai.com/mcp", "add [mcp_servers.docs] with the url"
    assert docs.get("required") is True, "mark the docs server required = true"
    assert "profiles" not in cfg and "profile" not in cfg, "remove legacy profile settings from config.toml"
    name, text = write_ci_profile()
    assert name == "ci.config.toml", f"profile file must be named 'ci.config.toml' (got {name!r})"
    prof = tomllib.loads(text)
    assert "profiles" not in prof, "use top-level keys in a profile file, not [profiles.ci]"
    assert prof.get("sandbox_mode") == "read-only", "CI profile: sandbox_mode = 'read-only'"
    assert prof.get("approval_policy") == "never", "CI profile: approval_policy = 'never'"
    assert prof.get("model") == "gpt-6-luna", "CI profile: model = 'gpt-6-luna'"

run_check("ex3", _check_ex3)

### Exercise 4 — Build `codex exec` argv for three CI tasks

Implement `build_ci_argv(task, prompt, model)` returning a list for `subprocess.run` (no shell):

| `task` | Requirements |
|---|---|
| `"review"` | `codex exec`, JSONL events, ephemeral, **read-only** sandbox, the given model, `--output-schema review.schema.json`, prompt last |
| `"autofix"` | `codex exec`, JSONL events, a sandbox that **allows edits in the workspace only**, the given model, prompt last |
| `"followup"` | continue the most recent non-interactive run: `codex exec resume --last`, JSONL events, prompt last |
| anything else | raise `ValueError` |

Never use removed or dangerous flags (`--full-auto`, `--dangerously-bypass-approvals-and-sandbox`, `--yolo`, `danger-full-access`). Short forms (`-s`, `-m`) are accepted. Note: `codex exec resume` does not take `--sandbox` (check `codex exec resume --help`).

[Theory](README.md#59-codex-exec-codex-in-ci-and-scripts)

In [ ]:
def build_ci_argv(task: str, prompt: str, model: str = "gpt-6-luna") -> list[str]:
    # TODO
    raise NotImplementedError

In [ ]:
def _flag(argv: list[str], *names: str):
    for i, a in enumerate(argv[:-1]):
        if a in names:
            return argv[i + 1]
    return None

def _check_ex4():
    BAD = {"--full-auto", "--dangerously-bypass-approvals-and-sandbox", "--yolo", "danger-full-access"}
    p = "Review the diff for bugs."
    r = build_ci_argv("review", p, "gpt-6-luna")
    assert isinstance(r, list) and all(isinstance(x, str) for x in r), "return a list of strings"
    assert r[:2] == ["codex", "exec"] and r[-1] == p, "review: start with codex exec, prompt last"
    assert _flag(r, "--sandbox", "-s") == "read-only", "review must be read-only"
    assert "--json" in r and "--ephemeral" in r, "review: JSONL events and ephemeral"
    assert _flag(r, "--model", "-m") == "gpt-6-luna", "pass the model"
    assert _flag(r, "--output-schema") == "review.schema.json", "review: --output-schema review.schema.json"
    a = build_ci_argv("autofix", "Fix lint errors.", "gpt-6.1-sol")
    assert a[:2] == ["codex", "exec"] and a[-1] == "Fix lint errors.", "autofix: codex exec ... prompt"
    assert _flag(a, "--sandbox", "-s") == "workspace-write", "autofix needs the workspace-write sandbox"
    assert "--json" in a and _flag(a, "--model", "-m") == "gpt-6.1-sol", "autofix: --json and the model"
    f = build_ci_argv("followup", "Now fix them.")
    assert f[:4] == ["codex", "exec", "resume", "--last"] and f[-1] == "Now fix them.", \
        "followup: codex exec resume --last ... prompt"
    assert "--json" in f, "followup: keep JSONL events"
    for argv in (r, a, f):
        assert not BAD & set(argv), f"remove removed/dangerous flags: {BAD & set(argv)}"
    try:
        build_ci_argv("deploy", "x")
    except ValueError:
        pass
    else:
        raise AssertionError("unknown task must raise ValueError")

run_check("ex4", _check_ex4)

### Exercise 5 — Summarize a `codex exec --json` event stream

Implement `summarize_events(jsonl)` returning a dict with:

- `thread_id` (from `thread.started`, else `None`)
- `final_message`: text of the **last completed** `agent_message` item (else `None`)
- `commands`: the `command` of each `command_execution` item from an `item.completed` event, in order, whatever its `status` (a failed command also arrives as `item.completed`; ignore `item.started`)
- `files_changed`: every `path` from `file_change` items whose `status` is `"completed"` (an `item.completed` event also reports **failed** patches, with `status: "failed"`; those changed nothing)
- `output_tokens`: sum of `usage.output_tokens` over `turn.completed` events (0 if none)
- `failed`: `True` if a `turn.failed` or a top-level `error` event appears (both mean the run did not succeed)

Skip blank lines and lines that are not valid JSON. [Theory](README.md#59-codex-exec-codex-in-ci-and-scripts)

In [ ]:
def summarize_events(jsonl: str) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
_OK_STREAM = '''
{"type":"thread.started","thread_id":"t-123"}
{"type":"turn.started"}
{"type":"item.started","item":{"id":"i1","type":"command_execution","command":"bash -lc 'make test'","status":"in_progress"}}
{"type":"item.completed","item":{"id":"i1","type":"command_execution","command":"bash -lc 'make test'","aggregated_output":"ok","exit_code":0,"status":"completed"}}
{"type":"item.completed","item":{"id":"i1b","type":"command_execution","command":"bash -lc 'make lint'","aggregated_output":"1 error","exit_code":1,"status":"failed"}}
warning: this line is not JSON

{"type":"item.completed","item":{"id":"i2","type":"file_change","changes":[{"path":"src/a.py","kind":"update"},{"path":"src/b.py","kind":"add"}],"status":"completed"}}
{"type":"item.completed","item":{"id":"i2b","type":"file_change","changes":[{"path":"never_written.py","kind":"add"}],"status":"failed"}}
{"type":"item.completed","item":{"id":"i3","type":"agent_message","text":"first draft"}}
{"type":"item.completed","item":{"id":"i4","type":"agent_message","text":"{\\"findings\\": []}"}}
{"type":"turn.completed","usage":{"input_tokens":1000,"cached_input_tokens":0,"output_tokens":42,"reasoning_output_tokens":0}}
'''
_FAIL_STREAM = '''
{"type":"thread.started","thread_id":"t-9"}
{"type":"turn.started"}
{"type":"turn.failed","error":{"message":"stream disconnected"}}
'''

def _check_ex5():
    s = summarize_events(_OK_STREAM)
    assert isinstance(s, dict), "return a dict"
    assert s.get("thread_id") == "t-123", "thread_id comes from thread.started"
    assert s.get("final_message") == '{"findings": []}', "final_message = text of the LAST completed agent_message"
    assert json.loads(s["final_message"]) == {"findings": []}, "with --output-schema the final message is a JSON string"
    assert s.get("commands") == ["bash -lc 'make test'", "bash -lc 'make lint'"], \
        "every command_execution from item.completed (failed ones too), none from item.started"
    assert "never_written.py" not in (s.get("files_changed") or []), "a file_change with status 'failed' changed nothing"
    assert s.get("files_changed") == ["src/a.py", "src/b.py"], "collect every path from completed file_change items"
    assert s.get("output_tokens") == 42, "sum usage.output_tokens from turn.completed"
    assert s.get("failed") is False, "a clean run is not failed"
    f = summarize_events(_FAIL_STREAM)
    assert f.get("failed") is True and f.get("final_message") is None, "turn.failed must set failed=True"
    assert summarize_events('{"type":"error","message":"fatal"}').get("failed") is True, \
        "a top-level error event is unrecoverable: set failed=True"
    e = summarize_events("")
    assert e.get("thread_id") is None and e.get("output_tokens") == 0 and e.get("commands") == [], "handle empty input"

run_check("ex5", _check_ex5)

## Part C — Architecture scenario

A fintech company (60 engineers, one Python monorepo with `services/payments`, `services/ledger` and `web/`) is rolling out Codex. Requirements:

1. Every engineer gets the same coding standards; the payments team has stricter rules and its own test command.
2. Codex must never run `terraform apply` or push to `main`, on any laptop.
3. Pull requests get an AI review that follows payments-specific review rules. The security team also wants machine-readable findings to feed their dashboard.
4. A nightly job should try to fix flaky lint errors and open a PR for a human to review.
5. Contractors work on the same repo from unmanaged laptops.

Design the Codex setup: which files go where (`AGENTS.md`, `config.toml`, profiles, skills, rules, hooks), which sandbox and approval settings apply on laptops vs. CI, how the PR review and the nightly job run on GitHub, and how you keep the API key and the repo safe.

**Your answer:**

_(write here)_

<details><summary>Rubric: what a strong answer contains</summary>

- **Instructions:** short root `AGENTS.md` (standards, `make test`, `## Code Review Rules`); `services/payments/AGENTS.override.md` (or nested `AGENTS.md`) with the payments test command and payments review rules (safe paths: Decimal or integer cents). No `@` imports; files kept well under the 32 KiB combined cap.
- **Enforcement, not prose:** `prefix_rule(pattern=["terraform","apply"], decision="forbidden")` and a `prompt`/`forbidden` rule for `git push` to main, enforced org-wide through admin `requirements.toml` (managed configuration), so it also binds contractors. Optionally a trusted `PreToolUse` hook for extra checks. States that `AGENTS.md` alone is guidance only.
- **Laptop defaults:** `workspace-write` + `on-request` (network off by default; scoped `network_access` + `network_proxy` allow rules for the package registry if needed); `.git`/`.codex`/`.agents` stay protected; no `danger-full-access` / `--yolo`. Requirements can disallow `approval_policy = "never"` and full access.
- **Contractors / untrusted contexts:** project `.codex/` layers load only for trusted projects; provider keys like `openai_base_url` are ignored at the project layer anyway; managed requirements apply regardless.
- **Workflows:** shared review/runbook procedures as repo skills in `.agents/skills/` (no custom prompts); side-effecting skills set `allow_implicit_invocation: false`.
- **PR review:** managed `@codex review` (or automatic reviews) for P0/P1 comments driven by the `AGENTS.md` review rules; plus `openai/codex-action@v1` with `sandbox: read-only` and `output-schema-file` for the security dashboard, a separate job with write permissions consumes `final-message`.
- **Nightly autofix:** Codex job with `contents: read`, `sandbox: workspace-write`, uploads a patch artifact; separate job with `contents: write` applies it and opens a PR without the key. `safety-strategy: drop-sudo` (default). Prompt treats repo/PR text as untrusted.
- **Keys:** `OPENAI_API_KEY` only as a secret passed to the action input (proxy); if raw `codex exec` is used elsewhere, `CODEX_API_KEY` inline for that process only, never job-wide; limit who can trigger workflows.
- **Profiles:** a `~/.codex/ci.config.toml` (e.g. `gpt-6-luna`, `read-only`, `never`) selected with `--profile ci` instead of `[profiles.ci]` tables. Required MCP servers marked `required = true`.
</details>

## Scorecard

In [ ]:
quiz_correct = QUIZ_SCORE["correct"]
ex_passed = sum(RESULTS.values())
total = QUIZ_SCORE["total"] + len(RESULTS)
combined = (quiz_correct + ex_passed) / total
print(f"Quiz:      {quiz_correct}/{QUIZ_SCORE['total']}")
print(f"Exercises: {ex_passed}/{len(RESULTS)}  {RESULTS}")
print(f"Combined:  {combined:.0%}  (pass mark 72%)")
print("PASS ✅" if combined >= 0.72 else "RETRY 🔁 — reread the README sections you missed, then try again")

---
## Solutions (spoilers)

Try everything first. Each block is collapsed.

<details><summary>Quiz answers with explanations</summary>

- **Q1** **B.** At global scope Codex reads `AGENTS.override.md` if it exists, otherwise `AGENTS.md`, and uses only the first non-empty file. Then it walks root → cwd, one file per directory. Closer files override by appearing later; they don't delete earlier files (D).

- **Q2** **C.** `untrusted` is no longer supported and can stop Codex from starting. The docs' migration is `read-only` + `on-request`, or `trust_level = "untrusted"` for the project to keep command approvals. `on-failure` is deprecated (A); there is no config cache (B); D removes every safeguard.

- **Q3** **A.** `codex exec` with an explicit read-only sandbox, JSONL events and a schema for the final message. B uses a flag that has been removed (the CLI rejects it) and relies on prose JSON. C is the interactive TUI. D removes the sandbox for a read-only task.

- **Q4** **D.** Profiles are separate files (`~/.codex/<name>.config.toml`) layered with `--profile`. `[profiles.*]` and `profile = ...` are legacy and rejected with a startup error (A). B would change laptops too (project config sits above the user config). C is guidance, not configuration.

- **Q5** **B.** Untrusted projects skip project `.codex/` layers entirely. And project configs can never set `openai_base_url` and other provider/credential keys; Codex ignores them with a warning.

- **Q6** **A.** Codex discovers repo skills in `.agents/skills/` (from cwd up to the repo root). AGENTS.md has no imports (B); custom prompts are deprecated and not shared (C); the file must be `SKILL.md` (D).

- **Q7** **C.** Keep the sandbox and grant the narrowest network access. A removes all boundaries. B removes prompts but grants nothing: the request just fails back to the model. D is guidance, not permission.

- **Q8** **D.** The documented pattern: the job that holds the key is read-only and only produces a patch; a separate job with write permissions applies it and never gets the key. A exposes the key job-wide and mixes it with write access. B disables the action's protections. C is the review feature; it does not push fixes.

- **Q9** **B.** The CLI, IDE extension and desktop app share MCP configuration from `config.toml`. `.mcp.json` is the Claude Code file (A). `codex mcp-server` was removed and is the opposite direction (C). D is not configuration.

- **Q10** **C.** The official Python SDK is `openai-codex` with `Codex`, `Sandbox`, `thread_start`, `run(output_schema=...)` and `thread_resume`. A is not the official package. B uses a removed command. D confuses the Agents SDK with Codex.

- **Q11** **A.** Scope and completion criteria: plan first, then a goal with an outcome, a constraint and runnable verification, plus a request for evidence. B adds capacity but no pass/fail signal. C is personal (not shared through the repo) and vague. D removes the safety controls and still doesn't define done.

- **Q12** **D.** Interacting issues (all through `parse_date`) go in one detailed message so Codex designs one consistent fix, with a specific failing case for each. The independent typo goes on its own. A churns on the date fixes. B mixes in unrelated work. C is vague feedback with no test cases.
</details>

<details><summary>Exercise 1 — AGENTS.md</summary>

```python
def write_agents_md() -> str:
    return """# AGENTS.md

## Repository expectations
- Python 3.11+. Run `make test` before you finish a task.
- Read configuration from environment variables; never commit secrets.
- Keep diffs small and focused on the task.

## Code Review Rules
### Payments
- Flag money arithmetic that uses float. Safe path: use Decimal or integer cents.
"""
```
</details>

<details><summary>Exercise 2 — resolve_chain</summary>

```python
def resolve_chain(tree: dict[str, dict[str, str]], cwd: str, global_files: dict[str, str],
                  fallbacks: tuple[str, ...] = (), max_bytes: int = 32 * 1024) -> list[str]:
    def pick(files: dict[str, str], names: tuple[str, ...]) -> str | None:
        for name in names:
            if files.get(name, "").strip():
                return name
        return None

    chain: list[str] = []
    g = pick(global_files, ("AGENTS.override.md", "AGENTS.md"))
    if g:
        chain.append(f"~/.codex/{g}")          # loaded separately: not part of the byte budget

    parts = [p for p in cwd.split("/") if p]
    dirs = [""] + ["/".join(parts[:i]) for i in range(1, len(parts) + 1)]
    remaining = max_bytes
    for d in dirs:
        if remaining <= 0:
            break
        files = tree.get(d, {})
        name = pick(files, ("AGENTS.override.md", "AGENTS.md", *fallbacks))
        if name:
            chain.append(f"{d}/{name}" if d else name)   # a file that crosses the cap is truncated, still loaded
            remaining -= min(len(files[name].encode("utf-8")), remaining)
    return chain
```
</details>

<details><summary>Exercise 3 — config.toml and CI profile</summary>

```python
def write_config_toml() -> str:
    return """
model = "gpt-6.1-sol"
approval_policy = "on-request"
sandbox_mode = "workspace-write"

[sandbox_workspace_write]
network_access = false

[mcp_servers.docs]
url = "https://developers.openai.com/mcp"
required = true
"""

def write_ci_profile() -> tuple[str, str]:
    return "ci.config.toml", """
model = "gpt-6-luna"
approval_policy = "never"
sandbox_mode = "read-only"
"""
```
</details>

<details><summary>Exercise 4 — build_ci_argv</summary>

```python
def build_ci_argv(task: str, prompt: str, model: str = "gpt-6-luna") -> list[str]:
    if task == "review":
        return ["codex", "exec", "--json", "--ephemeral", "--sandbox", "read-only", "--model", model,
                "--output-schema", "review.schema.json", prompt]
    if task == "autofix":
        return ["codex", "exec", "--json", "--sandbox", "workspace-write", "--model", model, prompt]
    if task == "followup":
        return ["codex", "exec", "resume", "--last", "--json", prompt]
    raise ValueError(f"unknown task: {task}")
```
</details>

<details><summary>Exercise 5 — summarize_events</summary>

```python
def summarize_events(jsonl: str) -> dict:
    out = {"thread_id": None, "final_message": None, "commands": [], "files_changed": [],
           "output_tokens": 0, "failed": False}
    for line in jsonl.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            ev = json.loads(line)
        except json.JSONDecodeError:
            continue
        t = ev.get("type")
        if t == "thread.started":
            out["thread_id"] = ev.get("thread_id")
        elif t == "item.completed":
            item = ev.get("item", {})
            kind = item.get("type")
            if kind == "agent_message":
                out["final_message"] = item.get("text")
            elif kind == "command_execution":
                out["commands"].append(item.get("command"))
            elif kind == "file_change" and item.get("status") == "completed":   # failed patches changed nothing
                out["files_changed"].extend(c.get("path") for c in item.get("changes", []))
        elif t == "turn.completed":
            out["output_tokens"] += ev.get("usage", {}).get("output_tokens", 0)
        elif t in ("turn.failed", "error"):
            out["failed"] = True
    return out
```
</details>

## Recap

- You can predict the `AGENTS.md` chain, write layered `config.toml` and profile files, and keep CI read-only unless edits are needed.
- You can build `codex exec` commands for CI and turn their JSONL output into facts a pipeline can act on.
- You can set up a Codex task with verifiable completion criteria, steer it, and pick the right follow-up strategy for interacting vs. independent issues.

Next chapter: [Chapter 6 — Prompt engineering and evals](../06-prompt-engineering-and-evals/README.md) · Back to the [chapter README](README.md)